encode ve decode yaz. decode(encode(metin)) metnin kendisini vermeli, Türkçe karakterli ve emoji içeren bir örnekle göster. Bazı token dizileri geçerli UTF-8 değil: decode bu durumda ne yapmalı, neden? (0:42:47 - 0:57:36)

### Tokenizer'ı eğit (task2'deki BPE)

In [1]:
with open('felatun_bey_tam_metin.txt', 'r', encoding='utf-8') as f:
    text = f.read()

In [2]:
def get_stats(ids):
    counts = {}
    for pair in zip(ids, ids[1:]):
        counts[pair] = counts.get(pair, 0) + 1
    return counts

In [3]:
def merge(ids, pair, idx):
    # ids içinde art arda gelen pair'leri yeni idx token'ı ile değiştir
    newids = []
    i = 0
    while i < len(ids):
        if i < len(ids) - 1 and ids[i] == pair[0] and ids[i+1] == pair[1]:
            newids.append(idx)
            i += 2
        else:
            newids.append(ids[i])
            i += 1
    return newids

In [4]:
vocab_size = 1000
num_merges = vocab_size - 256

tokens = list(text.encode("utf-8"))
ids = list(tokens)

merges = {} # (int, int) -> int
for i in range(num_merges):
    stats = get_stats(ids)
    pair = max(stats, key=stats.get)
    idx = 256 + i
    ids = merge(ids, pair, idx)
    merges[pair] = idx

vocab = {idx: bytes([idx]) for idx in range(256)}
for (p0, p1), idx in merges.items():
    vocab[idx] = vocab[p0] + vocab[p1]

print("byte sayısı :", len(tokens))
print("token sayısı:", len(ids))
print(f"sıkıştırma oranı: {len(tokens) / len(ids):.2f}X")

byte sayısı : 400857
token sayısı: 154399
sıkıştırma oranı: 2.60X


### decode
token id'leri → byte'lar → metin

In [5]:
def decode(ids):
    tokens = b"".join(vocab[idx] for idx in ids)
    text = tokens.decode("utf-8", errors="replace")
    return text

### encode
Metni byte'lara çevir, sonra `merges`'te **en önce öğrenilmiş** (en küçük idx'li) ikiliden başlayarak birleştir. Birleştirilecek ikili kalmayınca dur.

In [6]:
def encode(text):
    tokens = list(text.encode("utf-8"))
    while len(tokens) >= 2:
        stats = get_stats(tokens)
        pair = min(stats, key=lambda p: merges.get(p, float("inf")))
        if pair not in merges:
            break # birleştirilecek bir şey kalmadı
        idx = merges[pair]
        tokens = merge(tokens, pair, idx)
    return tokens

In [7]:
print(encode("merhaba"))
print(decode(encode("merhaba")))
print(encode("")) # boş metin
print(encode("a")) # tek karakter

[842, 676, 97]
merhaba
[]
[97]


### decode(encode(metin)) == metin

In [8]:
metin = "Çarşıda gördüğüm şövalye, ılık bir öğle güneşinde İstanbul'a doğru yürüdü 😊🚀🇹🇷 Ğ ü ş"
ids = encode(metin)
print("metin       :", metin)
print("byte sayısı :", len(metin.encode("utf-8")))
print("token sayısı:", len(ids))
print("token'lar   :", ids)
print()
print("decode      :", decode(ids))
print("aynı mı?    :", decode(encode(metin)) == metin)

metin       : Çarşıda gördüğüm şövalye, ılık bir öğle güneşinde İstanbul'a doğru yürüdü 😊🚀🇹🇷 Ğ ü ş
byte sayısı : 119
token sayısı: 61
token'lar   : [797, 265, 262, 256, 320, 452, 455, 269, 264, 277, 262, 283, 118, 278, 121, 705, 389, 361, 402, 283, 269, 333, 833, 655, 451, 413, 395, 268, 492, 39, 263, 594, 523, 291, 121, 386, 264, 455, 32, 240, 159, 152, 138, 240, 159, 154, 128, 240, 159, 135, 185, 240, 159, 135, 183, 32, 196, 158, 32, 728, 262]

decode      : Çarşıda gördüğüm şövalye, ılık bir öğle güneşinde İstanbul'a doğru yürüdü 😊🚀🇹🇷 Ğ ü ş
aynı mı?    : True


In [9]:
# her token'ın hangi byte'lara karşılık geldiği
print([vocab[i] for i in ids])

[b'\xc3\x87', b'ar', b'\xc5\x9f', b'\xc4\xb1', b'da ', b'g\xc3\xb6r', b'd\xc3\xbc', b'\xc4\x9f', b'\xc3\xbc', b'm ', b'\xc5\x9f', b'\xc3\xb6', b'v', b'al', b'y', b'e, ', b'\xc4\xb1l', b'\xc4\xb1k', b' bir ', b'\xc3\xb6', b'\xc4\x9f', b'le ', b'g\xc3\xbcn', b'e\xc5\x9f', b'inde ', b'\xc4\xb0', b'st', b'an', b'bul', b"'", b'a ', b'do', b'\xc4\x9fr', b'u ', b'y', b'\xc3\xbcr', b'\xc3\xbc', b'd\xc3\xbc', b' ', b'\xf0', b'\x9f', b'\x98', b'\x8a', b'\xf0', b'\x9f', b'\x9a', b'\x80', b'\xf0', b'\x9f', b'\x87', b'\xb9', b'\xf0', b'\x9f', b'\x87', b'\xb7', b' ', b'\xc4', b'\x9e', b' ', b'\xc3\xbc ', b'\xc5\x9f']


In [10]:
# eğitim metninin kendisi de birebir geri dönmeli
ornek = text[10000:12000]
print(decode(encode(ornek)) == ornek)

True


### Geçerli olmayan UTF-8 dizileri

In [11]:
# 128 tek başına geçerli bir UTF-8 başlangıç byte'ı değil (10xxxxxx = devam byte'ı)
try:
    print(b"".join(vocab[i] for i in [128]).decode("utf-8"))
except UnicodeDecodeError as e:
    print("strict decode hatası:", e)

print("errors='replace' ile:", repr(decode([128])))

strict decode hatası: 'utf-8' codec can't decode byte 0x80 in position 0: invalid start byte
errors='replace' ile: '�'


In [12]:
# 'ş' iki byte: [197, 159] -> sadece ilk byte'ı verirsek
print(list("ş".encode("utf-8")))
print(repr(decode([197])))
print(repr(decode([197, 159])))

# emoji 4 byte; yarısını verirsek
emoji_bytes = list("😊".encode("utf-8"))
print(emoji_bytes)
print(repr(decode(emoji_bytes[:2])))
print(repr(decode(emoji_bytes)))

[197, 159]
'�'
'ş'
[240, 159, 152, 138]
'�'
'😊'


In [13]:
# token'ları tek tek decode etmek ile hepsini birlikte decode etmek farklı
ids = encode("şöğüç 😊")
print("hepsi birlikte:", decode(ids))
print("tek tek       :", [decode([i]) for i in ids])

hepsi birlikte: şöğüç 😊
tek tek       : ['ş', 'ö', 'ğ', 'üç', ' ', '�', '�', '�', '�']


### Yorum

`decode(encode(metin)) == metin` → **True**. Türkçe harfler ve emojiler (hatta 🇹🇷 gibi iki kod noktasından oluşan bayrak) birebir geri geliyor, çünkü BPE sadece byte'ları birleştiriyor ve hiçbir bilgi kaybolmuyor; decode da birleştirmeyi tersine çevirip byte'ları art arda ekliyor.

**Geçersiz UTF-8 dizilerinde decode ne yapmalı?**
UTF-8'in bir kuralı var: çok byte'lı bir karakter `110xxxxx`/`1110xxxx`/`11110xxx` ile başlar ve ardından `10xxxxxx` devam byte'ları gelir. Tek başına `128` (`10000000`) ya da sadece `ş`'nin ilk byte'ı `197` gibi diziler bu kurala uymuyor. Python'un varsayılan `decode("utf-8")`'i (strict) bu durumda `UnicodeDecodeError` fırlatıyor.

Bence decode **hata fırlatmamalı**, `errors="replace"` ile geçersiz kısmı `�` (U+FFFD, replacement character) ile değiştirmeli:
- Dil modeli token'ları olasılığa göre örnekliyor; hangi token'ı üreteceğini garanti edemeyiz. Model bir karakterin sadece ilk byte'ını üretip sonra başka bir token'a geçebilir. Bu durumda programın çökmesi kabul edilemez.
- `�` görmek de bilgi veriyor: modelin geçerli olmayan bir şey ürettiğini anlıyoruz. `errors="ignore"` ise byte'ları sessizce silerdi, hatayı gizlerdi.
- Ayrıca token'ları **tek tek** decode etmemek gerekiyor: bir Türkçe harf ya da emoji iki farklı token'a bölünmüş olabilir. Önce tüm byte'ları birleştirip sonra decode edince doğru sonucu alıyoruz; tek tek yapınca yarım karakterler `�` oluyor (son hücrede görülüyor). Akış (streaming) halinde çıktı verirken de yarım kalan byte'ları tamponda bekletip karakter tamamlanınca yazdırmak gerekiyor.

**Not:** Örnekteki emojiler (😊🚀🇹🇷) ve `Ğ` eğitim metninde hiç (ya da sadece bir kez) geçmediği için hiçbir birleştirmeye girmedi; token listesinde ham byte'lar olarak kaldılar (`240, 159, 152, 138` ...). Yine de decode sorunsuz çalışıyor, çünkü vocab'ın ilk 256 elemanı tüm byte'ları kapsıyor: byte seviyesinde BPE'de "bilinmeyen karakter" (UNK) diye bir şey yok.